# 1.ipynb — Automated Utility & Telecom Billing Reconciliation System

**Capstone 01 — Configurable Rule Engine**

This notebook implements the deterministic foundation specified in the capstone:
- typed payment, billing AR, and dispute schemas
- immutable JSON state-delta telemetry
- sequential Priority Rules 1–6
- U01–U05 adjustment mapping
- $5.00 tolerance write-off
- UAC / UIC / QUERY exception routing
- LangGraph-compatible Supervisor workflow
- end-to-end test cases

> The notebook is intentionally self-contained for local testing. Azure/MCP integrations are represented by clean interfaces so they can be connected in later notebooks.

## Cell 1 — Environment setup

The project uses Python standard-library components for the deterministic core. LangGraph is optional during local development; if installed, the notebook uses it for graph execution.

In [ ]:
import os
import sys
import json
import re
import shutil
import logging
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from typing import Any, Dict, List, Optional, TypedDict, Literal, Tuple

PROJECT_ROOT = Path(".").resolve()
LOG_PATH = PROJECT_ROOT / "system.log"
CHROMA_PATH = PROJECT_ROOT / "chroma_db"

if LOG_PATH.exists():
    LOG_PATH.unlink()

print(f"Project root: {PROJECT_ROOT}")
print(f"Telemetry log: {LOG_PATH}")

## Cell 2 — Logging configuration

Create a console logger for developer feedback. The immutable audit stream itself is written by `JSONStateDeltaLogger` below.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("utility_reconciliation")

logger.info("Environment initialized")

## Cell 3 — Core constants

These constants preserve the six required end states, the six priority rules, the adjustment codes, and the $5.00 tolerance defined by the specification.

In [ ]:
END_STATES = {
    "OPEN",
    "PARTIAL MATCH",
    "CLOSED",
    "UAC",
    "UIC",
    "QUERY",
}

ADJUSTMENT_CODES = {
    "U01": "Meter Reading Dispute",
    "U02": "Service Outage Credit",
    "U03": "Estimated Bill Correction",
    "U04": "Assistance / Discount Adjustment",
    "U05": "Late Fee Waiver",
}

TOLERANCE_LIMIT = Decimal("5.00")

## Cell 4 — JSON state-delta logger

The specification requires immutable JSON telemetry in `system.log`, including timestamp, transaction ID, state transition, rule context, and payload delta.

In [ ]:
class JSONStateDeltaLogger:
    def __init__(self, path: Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log_state_delta(
        self,
        transaction_id: str,
        account_number: Optional[str],
        from_state: str,
        to_state: str,
        rule_context: Optional[Dict[str, Any]] = None,
        delta_payload: Optional[Dict[str, Any]] = None,
        guardrail_flag: Optional[bool] = None,
    ) -> Dict[str, Any]:
        event = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "telemetry_event": "STATE_DELTA_TRANSITION",
            "transaction_id": transaction_id,
            "account_number": account_number,
            "state_transition": {"from": from_state, "to": to_state},
            "rule_context": rule_context or {},
            "delta_payload": delta_payload or {},
        }
        if guardrail_flag is not None:
            event["guardrail_flag"] = guardrail_flag

        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(event, default=str) + "\n")

        return event

audit_logger = JSONStateDeltaLogger(LOG_PATH)
print("JSONStateDeltaLogger ready")

## Cell 5 — Logger smoke test

Write one sample transition matching the format shown in the capstone specification.

In [ ]:
audit_logger.log_state_delta(
    transaction_id="TXN-MER-6120",
    account_number="ACC-77210",
    from_state="OPEN",
    to_state="CLOSED",
    rule_context={"priority_rule": 4, "adjustment_code": None},
    delta_payload={"amount_paid": 482.50, "match_type": "2-Way"},
)

print(LOG_PATH.read_text(encoding="utf-8").splitlines()[-1])

## Cell 6 — Telemetry reader helper

A small helper makes audit validation and later governance tests straightforward.

In [ ]:
def read_telemetry_events(path: Path = LOG_PATH) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    events = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            events.append(json.loads(line))
    return events

print(f"Telemetry events: {len(read_telemetry_events())}")

## Cell 7 — Money normalization helper

Financial comparisons use `Decimal` rather than binary floating-point arithmetic.

In [ ]:
def money(value: Any) -> Decimal:
    return Decimal(str(value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

def money_float(value: Any) -> float:
    return float(money(value))

print(money("482.50"))

## Cell 8 — Environment and dependency verification

LangGraph is optional for the deterministic local implementation. The workflow logic remains directly executable even if the package is unavailable.

In [ ]:
try:
    import langgraph
    LANGGRAPH_AVAILABLE = True
    print("LangGraph detected:", getattr(langgraph, "__version__", "installed"))
except Exception:
    LANGGRAPH_AVAILABLE = False
    print("LangGraph not installed. Deterministic fallback execution will be used.")

## Cell 9 — Payment transaction schema

In [ ]:
@dataclass
class PaymentTransactionPayload:
    transaction_id: str
    account_number: Optional[str]
    amount_paid: Decimal
    reference_number: Optional[str] = None
    customer_name: Optional[str] = None
    meter_id: Optional[str] = None
    bill_date: Optional[str] = None

    def __post_init__(self):
        self.amount_paid = money(self.amount_paid)

## Cell 10 — Billing AR schema

In [ ]:
@dataclass
class BillingARPayload:
    bill_id: str
    account_number: str
    total_amount: Decimal
    meter_id: Optional[str] = None
    bill_date: Optional[str] = None

    def __post_init__(self):
        self.total_amount = money(self.total_amount)

## Cell 11 — Dispute note schema

In [ ]:
@dataclass
class DisputeNotePayload:
    dispute_id: str
    unstructured_notes: str
    adjustment_code: Optional[str] = None

## Cell 12 — Central reconciliation state

This is the state carried through the Supervisor-style workflow.

In [ ]:
class UtilityReconciliationState(TypedDict, total=False):
    payment_transaction: Optional[Dict[str, Any]]
    billing_ar: Optional[Dict[str, Any]]
    dispute_note: Optional[Dict[str, Any]]
    current_state: str
    matched_priority_rule: Optional[int]
    match_type: Optional[str]
    adjustment_code: Optional[str]
    adjustment_category: Optional[str]
    amount_paid: float
    bill_amount: float
    variance: float
    approved_settlement_amount: float
    held_dispute_variance: float
    auto_write_off_amount: float
    confidence: float
    requires_human_review: bool
    guardrail_triggered: bool
    threat_rules: List[str]
    audit_trail: List[str]

## Cell 13 — State factory

The initial state is `OPEN`, as required by the specification.

In [ ]:
def make_initial_state(
    payment: PaymentTransactionPayload,
    billing: Optional[BillingARPayload] = None,
    dispute: Optional[DisputeNotePayload] = None,
) -> UtilityReconciliationState:
    return {
        "payment_transaction": asdict(payment),
        "billing_ar": asdict(billing) if billing else None,
        "dispute_note": asdict(dispute) if dispute else None,
        "current_state": "OPEN",
        "matched_priority_rule": None,
        "match_type": None,
        "adjustment_code": None,
        "adjustment_category": None,
        "amount_paid": money_float(payment.amount_paid),
        "bill_amount": money_float(billing.total_amount) if billing else 0.0,
        "variance": 0.0,
        "approved_settlement_amount": 0.0,
        "held_dispute_variance": 0.0,
        "auto_write_off_amount": 0.0,
        "confidence": 0.0,
        "requires_human_review": False,
        "guardrail_triggered": False,
        "threat_rules": [],
        "audit_trail": [],
    }

## Cell 14 — State updater with telemetry

Every meaningful state transition is recorded in the immutable audit stream.

In [ ]:
def update_state_with_telemetry(
    state: UtilityReconciliationState,
    *,
    to_state: Optional[str] = None,
    rule_context: Optional[Dict[str, Any]] = None,
    delta_payload: Optional[Dict[str, Any]] = None,
    **updates,
) -> UtilityReconciliationState:
    previous = state.get("current_state", "OPEN")
    state.update(updates)

    if to_state is not None:
        state["current_state"] = to_state

    transaction = state.get("payment_transaction") or {}
    tx_id = transaction.get("transaction_id", "UNKNOWN")
    account = transaction.get("account_number")

    event = audit_logger.log_state_delta(
        transaction_id=tx_id,
        account_number=account,
        from_state=previous,
        to_state=state.get("current_state", previous),
        rule_context=rule_context,
        delta_payload=delta_payload or updates,
    )

    state.setdefault("audit_trail", []).append(
        f"{previous} -> {state.get('current_state')}"
        f" | rule={rule_context or {}}"
    )
    return state

## Cell 15 — Matching utilities

The priority engine evaluates the rules in strict sequence. A match means the identifying fields agree with the billing record; amount variance is handled separately.

In [ ]:
def same_amount(payment: Dict[str, Any], billing: Dict[str, Any]) -> bool:
    return money(payment["amount_paid"]) == money(billing["total_amount"])

def rule_1(payment, billing):
    return (
        payment.get("account_number") == billing.get("account_number")
        and payment.get("reference_number") == billing.get("bill_id")
        and same_amount(payment, billing)
    )

def rule_2(payment, billing):
    return (
        payment.get("account_number") == billing.get("account_number")
        and payment.get("meter_id")
        and payment.get("meter_id") == billing.get("meter_id")
        and same_amount(payment, billing)
    )

def rule_3(payment, billing):
    return (
        payment.get("account_number") == billing.get("account_number")
        and payment.get("reference_number") == billing.get("bill_id")
        and payment.get("bill_date")
        and payment.get("bill_date") == billing.get("bill_date")
        and same_amount(payment, billing)
    )

def rule_4(payment, billing):
    return (
        payment.get("account_number") == billing.get("account_number")
        and payment.get("account_number") is not None
    )

def rule_5(payment, billing, dispute):
    return (
        dispute is not None
        and payment.get("reference_number") == billing.get("bill_id")
        and payment.get("account_number") == billing.get("account_number")
        and payment.get("bill_date") == billing.get("bill_date")
    )

def rule_6(payment, billing, dispute):
    return (
        dispute is not None
        and payment.get("reference_number") == billing.get("bill_id")
        and payment.get("account_number") == billing.get("account_number")
    )

## Cell 16 — Priority-rule definitions

The capstone requires Rules 1–6 to be evaluated sequentially.

In [ ]:
PRIORITY_RULES = [
    (1, "2-Way", lambda p, b, d: rule_1(p, b)),
    (2, "2-Way", lambda p, b, d: rule_2(p, b)),
    (3, "2-Way", lambda p, b, d: rule_3(p, b)),
    (4, "2-Way", lambda p, b, d: rule_4(p, b)),
    (5, "3-Way", lambda p, b, d: rule_5(p, b, d)),
    (6, "3-Way", lambda p, b, d: rule_6(p, b, d)),
]

## Cell 17 — Priority engine

The engine identifies the first successful rule and calculates the monetary variance.

In [ ]:
def execute_priority_rules_node(
    state: UtilityReconciliationState
) -> UtilityReconciliationState:
    payment = state.get("payment_transaction")
    billing = state.get("billing_ar")
    dispute = state.get("dispute_note")

    if not payment:
        return update_state_with_telemetry(
            state,
            to_state="UIC",
            confidence=1.0,
            requires_human_review=False,
            delta_payload={"reason": "payment metadata absent"},
        )

    if not payment.get("account_number") and not payment.get("customer_name"):
        return update_state_with_telemetry(
            state,
            to_state="UIC",
            confidence=1.0,
            requires_human_review=False,
            delta_payload={"reason": "sender/account unidentified"},
        )

    if not billing:
        return update_state_with_telemetry(
            state,
            to_state="UAC",
            confidence=0.95,
            requires_human_review=False,
            delta_payload={"reason": "no billing AR record available"},
        )

    matched = None
    for priority, match_type, predicate in PRIORITY_RULES:
        try:
            if predicate(payment, billing, dispute):
                matched = (priority, match_type)
                break
        except Exception:
            continue

    if matched is None:
        return update_state_with_telemetry(
            state,
            to_state="QUERY",
            confidence=0.40,
            requires_human_review=True,
            delta_payload={"reason": "no priority rule matched"},
        )

    priority, match_type = matched
    paid = money(payment["amount_paid"])
    billed = money(billing["total_amount"])
    variance = billed - paid

    return update_state_with_telemetry(
        state,
        matched_priority_rule=priority,
        match_type=match_type,
        amount_paid=money_float(paid),
        bill_amount=money_float(billed),
        variance=money_float(variance),
        confidence=0.95 if priority <= 4 else 0.90,
        rule_context={"priority_rule": priority, "match_type": match_type},
        delta_payload={"amount_paid": money_float(paid), "bill_amount": money_float(billed)},
    )

## Cell 18 — Adjustment reason mapper

U01–U05 are mapped only from the supplied dispute/adjustment text. No unsupported reason is invented.

In [ ]:
ADJUSTMENT_PATTERNS = [
    ("U01", ["meter", "estimated", "high reading", "read appears"]),
    ("U02", ["outage", "service outage", "sla credit"]),
    ("U03", ["actual meter", "actual read", "re-read", "estimated bill correction"]),
    ("U04", ["senior", "low-income", "subsidy", "discount", "assistance"]),
    ("U05", ["late fee", "late-payment", "goodwill", "first-time occurrence"]),
]

def map_adjustment_reason(text: str) -> Tuple[Optional[str], Optional[str]]:
    normalized = text.lower()
    for code, terms in ADJUSTMENT_PATTERNS:
        if any(term in normalized for term in terms):
            return code, ADJUSTMENT_CODES[code]
    return None, None

## Cell 19 — Adjustment/discrepancy node

Short payment with supporting dispute evidence becomes `PARTIAL MATCH`; the paid amount is approved and the variance is held as the dispute amount.

In [ ]:
def adjustment_discrepancy_node(
    state: UtilityReconciliationState
) -> UtilityReconciliationState:
    variance = money(state.get("variance", 0))
    if variance <= 0:
        return state

    dispute = state.get("dispute_note")
    if not dispute:
        return state

    code, category = map_adjustment_reason(dispute.get("unstructured_notes", ""))
    if not code:
        return state

    paid = money(state["amount_paid"])

    return update_state_with_telemetry(
        state,
        to_state="PARTIAL MATCH",
        adjustment_code=code,
        adjustment_category=category,
        approved_settlement_amount=money_float(paid),
        held_dispute_variance=money_float(variance),
        rule_context={
            "priority_rule": state.get("matched_priority_rule"),
            "adjustment_code": code,
        },
        delta_payload={
            "approved_settlement_amount": money_float(paid),
            "held_dispute_variance": money_float(variance),
        },
    )

## Cell 20 — Tolerance auto-write-off node

Differences of $5.00 or less are automatically written off and the transaction is closed.

In [ ]:
def tolerance_auto_write_off_node(
    state: UtilityReconciliationState
) -> UtilityReconciliationState:
    variance = money(state.get("variance", 0))

    if variance <= 0:
        return update_state_with_telemetry(
            state,
            to_state="CLOSED",
            approved_settlement_amount=state.get("amount_paid", 0.0),
            confidence=max(state.get("confidence", 0.0), 0.95),
        )

    if variance <= TOLERANCE_LIMIT:
        return update_state_with_telemetry(
            state,
            to_state="CLOSED",
            auto_write_off_amount=money_float(variance),
            approved_settlement_amount=state.get("amount_paid", 0.0),
            confidence=max(state.get("confidence", 0.0), 0.95),
            delta_payload={"write_off": money_float(variance)},
        )

    return state

## Cell 21 — Unmatched exception node

This node preserves the specification's distinction:
- UAC = customer is recognized, bill is not identified
- UIC = sender/account is not identified
- QUERY = conflicting or low-confidence data

In [ ]:
def unmatched_exception_node(
    state: UtilityReconciliationState
) -> UtilityReconciliationState:
    payment = state.get("payment_transaction") or {}
    billing = state.get("billing_ar")

    if not payment.get("account_number") and not payment.get("customer_name"):
        return update_state_with_telemetry(
            state,
            to_state="UIC",
            confidence=1.0,
            requires_human_review=False,
            delta_payload={"classification": "UIC"},
        )

    if billing is None and (payment.get("account_number") or payment.get("customer_name")):
        return update_state_with_telemetry(
            state,
            to_state="UAC",
            confidence=0.95,
            requires_human_review=False,
            delta_payload={"classification": "UAC"},
        )

    if state.get("confidence", 0) < 0.85:
        return update_state_with_telemetry(
            state,
            to_state="QUERY",
            confidence=state.get("confidence", 0.0),
            requires_human_review=True,
            delta_payload={"classification": "QUERY"},
        )

    return state

## Cell 22 — Conditional routing logic

This is the Supervisor decision function used by the graph and the local fallback runner.

In [ ]:
def route_payment_workflow(state: UtilityReconciliationState) -> str:
    current = state.get("current_state")

    if current in {"CLOSED", "PARTIAL MATCH", "UAC", "UIC", "QUERY"}:
        return current

    if state.get("variance", 0) > 0:
        if state.get("dispute_note"):
            return "adjustment"
        return "tolerance"

    return "tolerance"

## Cell 23 — Supervisor workflow runner

The runner composes the deterministic nodes in the same conceptual order as the requested LangGraph Supervisor pattern.

In [ ]:
def run_reconciliation(
    payment: PaymentTransactionPayload,
    billing: Optional[BillingARPayload] = None,
    dispute: Optional[DisputeNotePayload] = None,
) -> UtilityReconciliationState:
    state = make_initial_state(payment, billing, dispute)

    state = execute_priority_rules_node(state)

    if state["current_state"] in {"UAC", "UIC", "QUERY"}:
        return state

    route = route_payment_workflow(state)

    if route == "adjustment":
        state = adjustment_discrepancy_node(state)
        if state["current_state"] == "PARTIAL MATCH":
            return state

    if route == "tolerance":
        state = tolerance_auto_write_off_node(state)
        if state["current_state"] == "CLOSED":
            return state

    return unmatched_exception_node(state)

## Cell 24 — LangGraph construction

When LangGraph is installed, construct a compiled graph. Otherwise retain the same runnable local workflow.

In [ ]:
def build_langgraph_app():
    if not LANGGRAPH_AVAILABLE:
        return None

    try:
        from langgraph.graph import StateGraph, END

        builder = StateGraph(UtilityReconciliationState)

        builder.add_node("supervisor_match", execute_priority_rules_node)
        builder.add_node("adjustment", adjustment_discrepancy_node)
        builder.add_node("tolerance", tolerance_auto_write_off_node)
        builder.add_node("exceptions", unmatched_exception_node)

        builder.set_entry_point("supervisor_match")

        def after_match(state):
            return route_payment_workflow(state)

        builder.add_conditional_edges(
            "supervisor_match",
            after_match,
            {
                "adjustment": "adjustment",
                "tolerance": "tolerance",
                "CLOSED": END,
                "PARTIAL MATCH": END,
                "UAC": END,
                "UIC": END,
                "QUERY": END,
            },
        )

        builder.add_edge("adjustment", END)
        builder.add_edge("tolerance", END)
        builder.add_edge("exceptions", END)

        return builder.compile()
    except Exception as exc:
        print("LangGraph graph construction skipped:", exc)
        return None

app = build_langgraph_app()
print("Compiled LangGraph app:", app is not None)

## Cell 25 — Test Case 1.1: Perfect 2-Way Match

Expected:
- Priority Rule: 4
- Final State: CLOSED
- Settlement: $482.50

This follows the capstone's positive example.

In [ ]:
t11 = run_reconciliation(
    PaymentTransactionPayload(
        transaction_id="TXN-11000",
        account_number="ACC-77210",
        amount_paid=482.50,
        reference_number="BILL-4471",
    ),
    BillingARPayload(
        bill_id="BILL-4471",
        account_number="ACC-77210",
        total_amount=482.50,
    ),
)

print({
    "Priority Rule Matched": t11["matched_priority_rule"],
    "Final State Output": t11["current_state"],
    "Approved Settlement Amount": t11["approved_settlement_amount"],
})

## Cell 26 — Test Case 1.2: Meter Reading Dispute

Expected:
- Adjustment Code: U01
- Approved Settlement: $560.00
- Held Dispute Variance: $50.00
- Final State: PARTIAL MATCH

In [ ]:
t12 = run_reconciliation(
    PaymentTransactionPayload(
        transaction_id="TXN-6055",
        account_number="ACC-77210",
        amount_paid=560.00,
        reference_number="BILL-4471",
    ),
    BillingARPayload(
        bill_id="BILL-4471",
        account_number="ACC-77210",
        total_amount=610.00,
    ),
    DisputeNotePayload(
        dispute_id="DSP-330",
        unstructured_notes="Meter read appears estimated high in error ($50 credit requested).",
    ),
)

print({
    "Assigned Adjustment Code": t12["adjustment_code"],
    "Approved Settlement Amount": t12["approved_settlement_amount"],
    "Held Dispute Variance": t12["held_dispute_variance"],
    "Final State": t12["current_state"],
})

## Cell 27 — Test Case 1.3: Tolerance Buffer

Expected:
- Auto Write-Off: $2.00
- Final State: CLOSED

In [ ]:
t13 = run_reconciliation(
    PaymentTransactionPayload(
        transaction_id="TXN-9002",
        account_number="ACC-88510",
        amount_paid=148.00,
        reference_number="BILL-2290",
    ),
    BillingARPayload(
        bill_id="BILL-2290",
        account_number="ACC-88510",
        total_amount=150.00,
    ),
)

print({
    "Auto Write-Off Amount": t13["auto_write_off_amount"],
    "Final System State": t13["current_state"],
})

## Cell 28 — Test Case 4.1: UAC and UIC negative cases

These validate the two cash exception classifications required by the specification.

In [ ]:
uac = run_reconciliation(
    PaymentTransactionPayload(
        transaction_id="TXN-5000",
        account_number=None,
        customer_name="Sunrise Apartments LLC",
        amount_paid=275.00,
    )
)

uic = run_reconciliation(
    PaymentTransactionPayload(
        transaction_id="TXN-UIC-001",
        account_number=None,
        customer_name=None,
        amount_paid=340.00,
    )
)

print("UAC:", uac["current_state"])
print("UIC:", uic["current_state"])

## Cell 29 — Batch execution helper

Run a collection of transactions and return compact final-state summaries.

In [ ]:
def summarize_state(state):
    return {
        "transaction_id": state["payment_transaction"]["transaction_id"],
        "state": state["current_state"],
        "priority_rule": state.get("matched_priority_rule"),
        "adjustment_code": state.get("adjustment_code"),
        "approved_settlement": state.get("approved_settlement_amount"),
        "variance": state.get("variance"),
        "human_review": state.get("requires_human_review"),
    }

batch = [t11, t12, t13, uac, uic]
for result in batch:
    print(summarize_state(result))

## Cell 30 — Assertions / automated verification

These assertions provide a deterministic acceptance gate for the foundational notebook.

In [ ]:
assert t11["matched_priority_rule"] == 4
assert t11["current_state"] == "CLOSED"
assert money(t11["approved_settlement_amount"]) == money("482.50")

assert t12["adjustment_code"] == "U01"
assert t12["current_state"] == "PARTIAL MATCH"
assert money(t12["approved_settlement_amount"]) == money("560.00")
assert money(t12["held_dispute_variance"]) == money("50.00")

assert money(t13["auto_write_off_amount"]) == money("2.00")
assert t13["current_state"] == "CLOSED"

assert uac["current_state"] == "UAC"
assert uic["current_state"] == "UIC"

print("All foundational assertions passed.")

## Cell 31 — Audit trail inspection

The specification requires every transition to be persisted in `system.log`.

In [ ]:
events = read_telemetry_events()

print(f"Total telemetry events logged: {len(events)}")
for event in events[-10:]:
    print(json.dumps(event, indent=2, default=str))

## Cell 32 — Final output verification

This cell summarizes the notebook status and the artifacts required by the capstone.

In [ ]:
required_files = [LOG_PATH]
print("1.ipynb foundational engine: READY")
print("system.log exists:", LOG_PATH.exists())
print("Telemetry event count:", len(read_telemetry_events()))
print("LangGraph available:", LANGGRAPH_AVAILABLE)
print()
print("Expected foundational states:")
print("  Perfect match -> CLOSED")
print("  Disputed short-pay -> PARTIAL MATCH / U01")
print("  <= $5 variance -> CLOSED with write-off")
print("  Recognized customer, no bill -> UAC")
print("  Unknown sender -> UIC")
print("  Conflicting/low-confidence case -> QUERY")